In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import ADASYN, SMOTE
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                            balanced_accuracy_score, confusion_matrix,
                            matthews_corrcoef, cohen_kappa_score, log_loss,
                            mean_squared_error, mean_absolute_error, r2_score)
import time
import os

# Set the number of K folds as a global variable
K_FOLDS = 5

# Read the dataset from CSV file
df = pd.read_csv('../../../datasets/dataset-1.csv')

# Print class distribution before resampling
print("Class distribution before resampling:")
print(df['GradeClass'].value_counts())
print()

# Rename the target column as 'label' for consistency with template
df.rename(columns={'GradeClass': 'label'}, inplace=True)

# Drop StudentID as it's not a relevant feature
X = df.drop(columns=['label', 'StudentID']).values
y = df['label'].values

timing_results = []

# Define resampling techniques
resampling_methods = {
    'None': None,
    'SMOTE': SMOTE(random_state=42),
    'ADASYN': ADASYN(random_state=42)
}

# Define classifiers
classifiers = {
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'KNN': KNeighborsClassifier(),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
}

# Store results
results = []

kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

# Create directories to save confusion matrices and class distribution plots
os.makedirs("confusion_matrices", exist_ok=True)
os.makedirs("class_distributions", exist_ok=True)

# Helper function for confusion matrix metrics
def confusion_matrix_metrics(cm, classes):
    metrics = {}
    for idx, class_label in enumerate(classes):
        TP = cm[idx, idx]  # True Positives for this class
        FP = cm[:, idx].sum() - TP  # False Positives for this class
        FN = cm[idx, :].sum() - TP  # False Negatives for this class
        TN = cm.sum() - (TP + FP + FN)  # True Negatives for this class

        metrics[class_label] = {
            'TPR': TP / (TP + FN + 1e-10) if (TP + FN) > 0 else 0,
            'TNR': TN / (TN + FP + 1e-10) if (TN + FP) > 0 else 0,
            'FPR': FP / (TN + FP + 1e-10) if (TN + FP) > 0 else 0,
            'FNR': FN / (TP + FN + 1e-10) if (TP + FN) > 0 else 0
        }
    return metrics

# Plot class distribution for each resampling method
def plot_class_distribution(y, method_name, fold_idx):
    plt.figure(figsize=(8, 6))
    unique, counts = np.unique(y, return_counts=True)
    plt.bar(unique.astype(str), counts)
    plt.title(f"Class Distribution - {method_name} - Fold {fold_idx}")
    plt.xlabel("Class")
    plt.ylabel("Frequency")
    plt.savefig(f"class_distributions/{method_name}_fold_{fold_idx}.png")
    plt.close()

# Calculate additional metrics
def calculate_additional_metrics(y_true, y_pred):
    metrics = {
        'Matthews Correlation Coefficient': matthews_corrcoef(y_true, y_pred),
        'Cohen Kappa Score': cohen_kappa_score(y_true, y_pred)
    }
    return metrics

# Iterate over resampling methods
for resample_name, resampler in resampling_methods.items():
    # Iterate over classifiers
    for clf_name, clf in classifiers.items():
        fold_idx = 1
        for train_index, test_index in kf.split(X):
            # Split the data
            X_train, X_test = X[train_index], X[test_index]
            y_train, y_test = y[train_index], y[test_index]
            
            # Apply resampling technique if specified
            if resampler is not None:
                try:
                    X_train, y_train = resampler.fit_resample(X_train, y_train)
                    print(f"Applied {resample_name} resampling for {clf_name} on fold {fold_idx}")
                    print(f"Class distribution after {resample_name}:")
                    unique, counts = np.unique(y_train, return_counts=True)
                    print(dict(zip(unique, counts)))
                    print()
                except Exception as e:
                    print(f"Error applying {resample_name} for {clf_name} on fold {fold_idx}: {e}")
                    continue
            
            # Plot the class distribution
            plot_class_distribution(y_train, f"{resample_name}_{clf_name}", fold_idx)

            # Record start time
            start_train_time = time.time()
            clf.fit(X_train, y_train)
            train_time = time.time() - start_train_time

            start_test_time = time.time()
            y_pred = clf.predict(X_test)
            test_time = time.time() - start_test_time

            timing_results.append({
                'Resampling': resample_name,
                'Classifier': clf_name,
                'Fold': fold_idx,
                'Training Time (s)': train_time,
                'Testing Time (s)': test_time,
                'Total Time (s)': train_time + test_time
            })

            # Compute metrics
            unique_classes = np.unique(np.concatenate((y_train, y_test)))
            cm = confusion_matrix(y_test, y_pred, labels=unique_classes)
            cm_metrics = confusion_matrix_metrics(cm, unique_classes)
            
            # Calculate additional metrics for the entire test set
            additional_metrics = calculate_additional_metrics(y_test, y_pred)

            class_metrics_list = []

            # Calculate overall metrics
            overall_metrics = {
                'Resampling': resample_name,
                'Classifier': clf_name,
                'Fold': fold_idx,
                'Class': 'Overall',
                'Accuracy': accuracy_score(y_test, y_pred),
                'Precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
                'Recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
                'F1 Score': f1_score(y_test, y_pred, average='weighted', zero_division=0),
                'Balanced Accuracy': balanced_accuracy_score(y_test, y_pred),
                'Matthews Correlation': additional_metrics['Matthews Correlation Coefficient'],
                'Cohen Kappa': additional_metrics['Cohen Kappa Score'],
                'Training Time (s)': train_time,
                'Testing Time (s)': test_time
            }
            class_metrics_list.append(overall_metrics)

            for class_label in unique_classes:
                class_mask_test = (y_test == class_label)
                if class_mask_test.sum() == 0:
                    # Skip classes with no instances in the test set for this fold
                    class_specific_metrics = {
                        'Resampling': resample_name,
                        'Classifier': clf_name,
                        'Fold': fold_idx,
                        'Class': class_label,
                        'Accuracy': np.nan,
                        'Precision': np.nan,
                        'Recall': np.nan,
                        'F1 Score': np.nan,
                        'Balanced Accuracy': np.nan,
                        'True Positive Rate (TPR)': np.nan,
                        'True Negative Rate (TNR)': np.nan,
                        'False Positive Rate (FPR)': np.nan,
                        'False Negative Rate (FNR)': np.nan,
                        'Training Time (s)': train_time,
                        'Testing Time (s)': test_time
                    }
                else:
                    class_specific_metrics = {
                        'Resampling': resample_name,
                        'Classifier': clf_name,
                        'Fold': fold_idx,
                        'Class': class_label,
                        'True Positive Rate (TPR)': cm_metrics[class_label]['TPR'],
                        'True Negative Rate (TNR)': cm_metrics[class_label]['TNR'],
                        'False Positive Rate (FPR)': cm_metrics[class_label]['FPR'],
                        'False Negative Rate (FNR)': cm_metrics[class_label]['FNR'],
                        'Training Time (s)': train_time,
                        'Testing Time (s)': test_time
                    }

                class_metrics_list.append(class_specific_metrics)

            # Plot and save confusion matrix
            plt.figure(figsize=(8, 6))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=unique_classes, yticklabels=unique_classes)
            plt.title(f"{resample_name} - {clf_name} - Fold {fold_idx} Confusion Matrix")
            plt.xlabel("Predicted")
            plt.ylabel("True")
            plt.savefig(f"confusion_matrices/{resample_name}_{clf_name}_fold_{fold_idx}.png")
            plt.close()

            # Append results for this fold
            results.extend(class_metrics_list)
            fold_idx += 1

# Create DataFrames for results and timing
timing_df = pd.DataFrame(timing_results)
timing_df.to_csv("time_comparison.csv", index=False)

# Create a DataFrame for results
results_df = pd.DataFrame(results)
print("Classification Metrics Across Resampling Methods and Folds:")
print(results_df.head())

# Save results to CSV
results_df.to_csv("metrics_comparison.csv", index=False)

# Generate summary statistics
summary_df = results_df[results_df['Class'] == 'Overall'].groupby(['Resampling', 'Classifier']).agg({
    'Accuracy': ['mean', 'std'],
    'Precision': ['mean', 'std'],
    'Recall': ['mean', 'std'],
    'F1 Score': ['mean', 'std'],
    'Balanced Accuracy': ['mean', 'std'],
    'Matthews Correlation': ['mean', 'std'],
    'Cohen Kappa': ['mean', 'std'],
    'Training Time (s)': ['mean', 'sum'],
    'Testing Time (s)': ['mean', 'sum']
})

summary_df.to_csv("summary_metrics.csv")

# Plot comparison of resampling techniques across classifiers
plt.figure(figsize=(15, 10))
sns.barplot(x='Classifier', y='F1 Score', hue='Resampling', data=results_df[results_df['Class'] == 'Overall'])
plt.title('F1 Score Comparison Across Resampling Methods and Classifiers')
plt.savefig('f1_comparison.png')
plt.close()

plt.figure(figsize=(15, 10))
sns.barplot(x='Classifier', y='Balanced Accuracy', hue='Resampling', data=results_df[results_df['Class'] == 'Overall'])
plt.title('Balanced Accuracy Comparison Across Resampling Methods and Classifiers')
plt.savefig('balanced_accuracy_comparison.png')
plt.close()

# Plot timing information
plt.figure(figsize=(15, 10))
sns.barplot(x='Classifier', y='Training Time (s)', hue='Resampling', data=timing_df)
plt.title('Training Time Comparison Across Resampling Methods and Classifiers')
plt.savefig('training_time_comparison.png')
plt.close()

print("Analysis complete. Check CSV files and visualizations for detailed results.")

Class distribution before resampling:
GradeClass
4.0    1211
3.0     414
2.0     391
1.0     269
0.0     107
Name: count, dtype: int64



/home/yashbuddhadev21/.local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/yashbuddhadev21/.local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html

Applied SMOTE resampling for DecisionTree on fold 1
Class distribution after SMOTE:
{0.0: 974, 1.0: 974, 2.0: 974, 3.0: 974, 4.0: 974}

Applied SMOTE resampling for DecisionTree on fold 2
Class distribution after SMOTE:
{0.0: 964, 1.0: 964, 2.0: 964, 3.0: 964, 4.0: 964}

Applied SMOTE resampling for DecisionTree on fold 3
Class distribution after SMOTE:
{0.0: 958, 1.0: 958, 2.0: 958, 3.0: 958, 4.0: 958}

Applied SMOTE resampling for DecisionTree on fold 4
Class distribution after SMOTE:
{0.0: 958, 1.0: 958, 2.0: 958, 3.0: 958, 4.0: 958}

Applied SMOTE resampling for DecisionTree on fold 5
Class distribution after SMOTE:
{0.0: 990, 1.0: 990, 2.0: 990, 3.0: 990, 4.0: 990}

Applied SMOTE resampling for KNN on fold 1
Class distribution after SMOTE:
{0.0: 974, 1.0: 974, 2.0: 974, 3.0: 974, 4.0: 974}

Applied SMOTE resampling for KNN on fold 2
Class distribution after SMOTE:
{0.0: 964, 1.0: 964, 2.0: 964, 3.0: 964, 4.0: 964}

Applied SMOTE resampling for KNN on fold 3
Class distribution afte

/home/yashbuddhadev21/.local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Applied SMOTE resampling for LogisticRegression on fold 2
Class distribution after SMOTE:
{0.0: 964, 1.0: 964, 2.0: 964, 3.0: 964, 4.0: 964}



...

Applied ADASYN resampling for LogisticRegression on fold 4
Class distribution after ADASYN:
{0.0: 951, 1.0: 933, 2.0: 925, 3.0: 954, 4.0: 958}



/home/yashbuddhadev21/.local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Applied ADASYN resampling for LogisticRegression on fold 5
Class distribution after ADASYN:
{0.0: 1002, 1.0: 1001, 2.0: 969, 3.0: 991, 4.0: 990}



/home/yashbuddhadev21/.local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Classification Metrics Across Resampling Methods and Folds:
  Resampling    Classifier  Fold    Class  Accuracy  Precision    Recall  \
0       None  DecisionTree     1  Overall  0.832985   0.839586  0.832985   
1       None  DecisionTree     1      0.0       NaN        NaN       NaN   
2       None  DecisionTree     1      1.0       NaN        NaN       NaN   
3       None  DecisionTree     1      2.0       NaN        NaN       NaN   
4       None  DecisionTree     1      3.0       NaN        NaN       NaN   

   F1 Score  Balanced Accuracy  Matthews Correlation  Cohen Kappa  \
0  0.834317           0.777021              0.755799     0.754984   
1       NaN                NaN                   NaN          NaN   
2       NaN                NaN                   NaN          NaN   
3       NaN                NaN                   NaN          NaN   
4       NaN                NaN                   NaN          NaN   

   Training Time (s)  Testing Time (s)  True Positive Rate (TPR)  \
